# AI Usage

AI tools(Chatgpt, Claude) were used only for assistance in designing the user interface (UI) of the project. The hashing logic, algorithm implementation, and data preprocessing were developed by the members of the group.

# Algorithmic Scope

This project focuses on performing a large-scale analysis of a Spotify songs dataset containing approximately 1.2 million song records using hashing as the main algorithmic technique, primarily implemented through Python dictionaries for fast indexing and lookup. In particular, we will apply concepts related to feature hashing, where metadata such as song titles, artist names, and other attributes are indexed to compute frequency counts and metadata associations across the dataset.

### Dataset

**Defining the dataset** 

In [ ]:
import pandas as pd
file_path = r"C:\HW3-Extra credit\tracks_features.csv"
df = pd.read_csv(file_path)
print("Shape:", df.shape)
print(df.head(2))
print(df.columns)


Shape: (1204025, 24)
                       id             name                      album  \
0  7lmeHLHBe4nmXzuXc0HDjk          Testify  The Battle Of Los Angeles   
1  1wsRitfRRtWyEapl0q22o8  Guerrilla Radio  The Battle Of Los Angeles   

                 album_id                       artists  \
0  2eia0myWFgoHuttJytCxgX  ['Rage Against The Machine']   
1  2eia0myWFgoHuttJytCxgX  ['Rage Against The Machine']   

                   artist_ids  track_number  disc_number  explicit  \
0  ['2d0hyoQ5ynDBnkvAbJKORj']             1            1     False   
1  ['2d0hyoQ5ynDBnkvAbJKORj']             2            1      True   

   danceability  ...  speechiness  acousticness  instrumentalness  liveness  \
0         0.470  ...       0.0727        0.0261          0.000011     0.356   
1         0.599  ...       0.1880        0.0129          0.000071     0.155   

   valence    tempo  duration_ms  time_signature  year  release_date  
0    0.503  117.906       210133             4.0  1999    199

The CSV file contains around 1204025 rows and 24 columns, including multiple metadata fields and unfiltered values.

# Data Preprocessing

The preprocessing step includes
* Converting data types (such as lists to strings).
* Inspecting missing values and duplicates
* Dropping unnecessary columns
* Handling null values
* Missing data is considered under    different cases such as MAR and MNAR


Dropping Un-necessary columns

In [ ]:
columns_to_drop = [
    "id",
    "duration_ms",
    "mode",
    "disc_number",
    "time_signature",
    "key"
]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("Remaining columns:")
print(df.columns)

Remaining columns:
Index(['name', 'album', 'album_id', 'artists', 'artist_ids', 'track_number',
       'explicit', 'danceability', 'energy', 'loudness', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'year', 'release_date'],
      dtype='object')


Identifying Duplicate entries

In [ ]:

print("Duplicate rows:", df.duplicated().sum())


duplicates = df[df.duplicated()]
print(duplicates.head())

Duplicate rows: 0
Empty DataFrame
Columns: [name, album, album_id, artists, artist_ids, track_number, explicit, danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, year, release_date]
Index: []


Since there are no duplicated rows, we dont drop any 

But after performing some EDA we found out "release_date" and "year" were column duplicates so we dropped the column=="release date"


In [ ]:
df = df.drop(columns=["release_date"], errors="ignore")

### Checking for missing values 

In [ ]:
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
name                 3
album               11
album_id             0
artists              0
artist_ids           0
track_number         0
explicit             0
danceability         0
energy               0
loudness             0
speechiness          0
acousticness         0
instrumentalness     0
liveness             0
valence              0
tempo                0
year                 0
dtype: int64


In [ ]:
import numpy as np

df["name"] = df["name"].fillna(np.nan)
df["album"] = df["album"].fillna(np.nan)

Handling empty strings

In [ ]:
df["name"] = df["name"].replace("", np.nan)
df["album"] = df["album"].replace("", np.nan)

The missing values in the name and album columns were retained as NaN to preserve the original data without introducing assumptions or modifications.

### Normalize text and converting data types

The artists column was transformed from list-like values into structured text and count-based features. This made it possible to identify solo and collaborative tracks, support metadata associations, and use artist information more effectively for indexing and frequency analysis.

In [ ]:
import ast


df["name"] = df["name"].str.lower().str.strip()
df["album"] = df["album"].str.lower().str.strip()
df["artists"] = df["artists"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df["artists"] = df["artists"].apply(
    lambda x: [artist.lower().strip() for artist in x]
)

In [ ]:
print(df["artists"].iloc[0])
print(type(df["artists"].iloc[0]))

['rage against the machine']
<class 'list'>


In [ ]:
df["track_type"] = df["artists"].apply(
    lambda x: "individual" if len(x) == 1 else "feature"
)

The lambda function is applied to each row of the artists column, where each entry is already a list of artists. For each list, it checks its length, and if the list has one element it labels it as "solo", otherwise if it has multiple elements it labels it as "collaboration"

This code first converts each value in the artists column into an actual Python list, so entries like ['Artist A', 'Artist B'] can be handled as separate artist names instead of one long string. It then uses those lists to create a track_type label based on how many artists are present, and builds two mappings where each artist is connected to all related songs and unique albums

The above step was mainly done so that each artist could be accessed individually, along with creating additional features such as track type for distinguishing between solo and collaborative tracks. This made the data usable for indexing and building relationships between artists, songs, and albums.

Store the CSV file 

In [ ]:
df.to_csv("processed_spotify_data.csv", index=False)

### ML implementation

An average feature column

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

FEATURE_COLS = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

# ── Step 1: Scale ──────────────────────────────────────────────────────────
df_clean = df.dropna(subset=FEATURE_COLS).copy()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURE_COLS])

# ── Step 2: K-Means as a pre-filter only (not the final ranker) ────────────
# More clusters = tighter pre-filter = faster similarity search
kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
df_clean["cluster"] = kmeans.fit_predict(X_scaled)

# Store scaled vectors back for similarity lookup
df_clean["_vec"] = list(X_scaled)

def get_similar_songs(song_name, df, limit=10, expand_clusters=1):
    name_norm = song_name.lower().strip()

    # ── fuzzy match on name ───────────────────────────────────────────────
    match = df[df["name"].str.lower().str.strip() == name_norm]
    if match.empty:
        # partial match fallback
        match = df[df["name"].str.lower().str.contains(name_norm, na=False)]
    if match.empty:
        print(f"Song not found: '{song_name}'")
        return pd.DataFrame()

    query_row    = match.iloc[0]
    query_vec    = np.array(query_row["_vec"]).reshape(1, -1)
    query_cluster = query_row["cluster"]

    
    if expand_clusters > 0:
        
        centroid_q   = kmeans.cluster_centers_[query_cluster].reshape(1, -1)
        centroid_sim = cosine_similarity(centroid_q, kmeans.cluster_centers_)[0]
        top_clusters = np.argsort(centroid_sim)[::-1][:1 + expand_clusters]
        candidates   = df[df["cluster"].isin(top_clusters)]
    else:
        candidates = df[df["cluster"] == query_cluster]

    
    candidates = candidates[
        candidates["name"].str.lower().str.strip() != name_norm
    ].copy()

    if candidates.empty:
        print("No candidates found in cluster.")
        return pd.DataFrame()

    # ── cosine similarity ranking ─────────────────────────────────────────
    candidate_vecs = np.vstack(candidates["_vec"].values)
    sims           = cosine_similarity(query_vec, candidate_vecs)[0]
    candidates     = candidates.copy()
    candidates["similarity"] = sims

    # ── return top results ────────────────────────────────────────────────
    result = (
        candidates
        .sort_values("similarity", ascending=False)
        .head(limit)[["name", "artists", "album", "year", "cluster", "similarity"]]
        .reset_index(drop=True)
    )
    result.index += 1  # start rank from 1
    return result


# ── Usage ──────────────────────────────────────────────────────────────────
results = get_similar_songs("gods plan", df_clean, limit=20)
print(results.to_string())

                                    name                                           artists                                album  year  cluster  similarity
1                    more than a miracle                                     [uglyography]               undercover new machine  2011       13    0.992413
2              speakin' to that mountain                                    [becky buller]                    crêpe paper heart  2018       13    0.992391
3                         follow through                                         [low fat]                           bar wisdom  2005       13    0.991714
4                 faith (hebrews 11:1,6)                            [seeds family worship]                       seeds of faith  2004       13    0.990444
5                       john d. champion                                    [becky buller]                    crêpe paper heart  2018       13    0.989811
6               expressway to your heart                              

# Data Analysis

Feature hashing is mainly used as part of the data analysis by using Python dictionaries to connect artists, songs, and albums, making it easier to search and understand relationships within the dataset.

In this part of the analysis well be doing Metadata relationship analysis

1) artist → songs
2) artist → albums
3) album → songs
4) song → artists
5) year → songs

Establishing relationships among millions of entities related to artists and albums and songs.

In [ ]:
import pandas as pd
import ast
import requests
import base64
import os
from difflib import get_close_matches

# ==============================
# 1. SPOTIFY API
# ==============================
CLIENT_ID = "d941bc8a2cbb466a8356fa1440c9e16e"
CLIENT_SECRET = "021d375ad4f7499ab92b155d476effe7"

def get_token():
    auth_str = f"{CLIENT_ID}:{CLIENT_SECRET}"
    b64_auth = base64.b64encode(auth_str.encode()).decode()

    url = "https://accounts.spotify.com/api/token"
    headers = {"Authorization": f"Basic {b64_auth}"}
    data = {"grant_type": "client_credentials"}

    response = requests.post(url, headers=headers, data=data)
    response.raise_for_status()
    return response.json()["access_token"]

def spotify_fetch_by_id(item_id, fetch_type="artist", token=None):
    url = f"https://api.spotify.com/v1/{fetch_type}s/{item_id}"
    headers = {"Authorization": f"Bearer {token}"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()

    if fetch_type == "artist":
        return {
            "name": data.get("name"),
            "id": data.get("id"),
            "image": data["images"][0]["url"] if data.get("images") else None
        }

    elif fetch_type == "album":
        return {
            "name": data.get("name"),
            "id": data.get("id"),
            "image": data["images"][0]["url"] if data.get("images") else None
        }

    elif fetch_type == "track":
        return {
            "name": data.get("name"),
            "id": data.get("id"),
            "album": data.get("album", {}).get("name"),
            "image": data.get("album", {}).get("images", [{}])[0].get("url")
                     if data.get("album", {}).get("images") else None
        }

    return data

# ==============================
# 2. CLEAN LIST-LIKE COLUMNS
# ==============================
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except:
            return [x]
    return [x]

# df = pd.read_csv("your_file.csv")   # load your CSV before running below

df["artists"] = df["artists"].apply(ensure_list)
df["artist_ids"] = df["artist_ids"].apply(ensure_list)

df["name"] = df["name"].astype(str).str.lower().str.strip()
df["album"] = df["album"].astype(str).str.lower().str.strip()
df["artists"] = df["artists"].apply(lambda x: [str(a).lower().strip() for a in x])
df["artist_ids"] = df["artist_ids"].apply(lambda x: [str(a).strip() for a in x])

# optional: make album_id safe too
df["album_id"] = df["album_id"].astype(str).str.strip()

# ==============================
# 3. BUILD HASH RELATIONSHIPS
# ==============================
artist_to_songs = {}
artist_to_id = {}

for _, row in df.iterrows():
    artists = row["artists"]
    artist_ids = row["artist_ids"]
    song = row["name"]
    album = row["album"]
    album_id = row["album_id"]
    year = row["year"]

    for i, artist in enumerate(artists):
        artist = artist.lower().strip()
        current_artist_id = artist_ids[i] if i < len(artist_ids) else None

        if artist not in artist_to_songs:
            artist_to_songs[artist] = {
                "solo": [],
                "main": [],
                "feature": []
            }

        if artist not in artist_to_id and current_artist_id:
            artist_to_id[artist] = current_artist_id

        if len(artists) == 1:
            role = "solo"
        elif i == 0:
            role = "main"
        else:
            role = "feature"

        artist_to_songs[artist][role].append({
            "song": song,
            "album": album,
            "album_id": album_id,
            "year": year
        })

# ==============================
# 4. FUZZY SEARCH
# ==============================
all_artists = list(artist_to_songs.keys())

def search_artist(user_input):
    user_input = user_input.lower().strip()

    if user_input in artist_to_songs:
        return user_input

    matches = get_close_matches(user_input, all_artists, n=1, cutoff=0.6)
    if matches:
        return matches[0]

    for artist in all_artists:
        if user_input in artist or artist in user_input:
            return artist

    return None

# ==============================
# 5. HTML DISPLAY
# ==============================
def show_artist_songs_html(user_input, output_file="artist_results.html"):
    matched_artist = search_artist(user_input)

    if not matched_artist:
        print("Artist not found.")
        return

    artist_id = artist_to_id.get(matched_artist)
    if not artist_id:
        print("No artist ID found in CSV for this artist.")
        return

    token = get_token()

    try:
        artist_info = spotify_fetch_by_id(artist_id, fetch_type="artist", token=token)
        artist_name = artist_info["name"] if artist_info else matched_artist.title()
        artist_image = artist_info["image"] if artist_info else None
    except:
        artist_name = matched_artist.title()
        artist_image = None

    grouped_songs = {"solo": [], "main": [], "feature": []}

    for role in ["solo", "main", "feature"]:
        for item in artist_to_songs[matched_artist][role]:
            album_cover = None

            if item["album_id"] and item["album_id"] != "nan":
                try:
                    album_info = spotify_fetch_by_id(item["album_id"], fetch_type="album", token=token)
                    if album_info:
                        album_cover = album_info["image"]
                except:
                    album_cover = None

            grouped_songs[role].append({
                "song": item["song"],
                "year": item["year"],
                "album": item["album"],
                "album_cover": album_cover
            })

    def build_song_cards(song_list):
        if not song_list:
            return '<p class="empty">No songs found in this category.</p>'

        cards = []
        for song in song_list:
            cover_html = (
                f'<img src="{song["album_cover"]}" class="album-cover" alt="Album Cover">'
                if song["album_cover"] else
                '<div class="album-cover placeholder">No Image</div>'
            )

            album_name = song["album"].title() if song["album"] else "Unknown"
            song_name = song["song"].title() if song["song"] else "Unknown"

            cards.append(f"""
            <div class="song-card">
                {cover_html}
                <div class="song-info">
                    <div class="song-title">{song_name}</div>
                    <div class="song-meta">Album: {album_name}</div>
                    <div class="song-meta">Year: {song["year"]}</div>
                </div>
            </div>
            """)

        return "\n".join(cards)

    artist_img_html = (
        f'<img src="{artist_image}" class="artist-image" alt="Artist Image">'
        if artist_image else
        '<div class="artist-image placeholder">No Image</div>'
    )

    html = f"""
    <html>
    <head>
        <title>{artist_name} - Songs</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                background: #121212;
                color: white;
                margin: 0;
                padding: 0;
            }}

            .container {{
                max-width: 1200px;
                margin: auto;
                padding: 30px;
            }}

            .artist-header {{
                display: flex;
                align-items: center;
                gap: 25px;
                background: #1e1e1e;
                padding: 25px;
                border-radius: 20px;
                margin-bottom: 30px;
                box-shadow: 0 4px 20px rgba(0,0,0,0.3);
            }}

            .artist-image {{
                width: 180px;
                height: 180px;
                object-fit: cover;
                border-radius: 50%;
                border: 4px solid #1db954;
            }}

            .placeholder {{
                display: flex;
                align-items: center;
                justify-content: center;
                background: #333;
                color: #bbb;
            }}

            .artist-name {{
                font-size: 42px;
                font-weight: bold;
                margin-bottom: 10px;
            }}

            .artist-sub {{
                color: #b3b3b3;
                font-size: 18px;
            }}

            .section {{
                margin-top: 35px;
            }}

            .section h2 {{
                font-size: 28px;
                border-left: 6px solid #1db954;
                padding-left: 12px;
                margin-bottom: 20px;
            }}

            .songs-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fill, minmax(320px, 1fr));
                gap: 20px;
            }}

            .song-card {{
                background: #1a1a1a;
                border-radius: 18px;
                overflow: hidden;
                box-shadow: 0 4px 14px rgba(0,0,0,0.25);
                display: flex;
                gap: 15px;
                padding: 15px;
                align-items: center;
            }}

            .album-cover {{
                width: 90px;
                height: 90px;
                object-fit: cover;
                border-radius: 12px;
                flex-shrink: 0;
            }}

            .song-info {{
                flex: 1;
            }}

            .song-title {{
                font-size: 19px;
                font-weight: bold;
                margin-bottom: 8px;
            }}

            .song-meta {{
                color: #cfcfcf;
                font-size: 14px;
                margin-bottom: 4px;
            }}

            .empty {{
                color: #999;
                font-style: italic;
            }}
        </style>
    </head>
    <body>
        <div class="container">

            <div class="artist-header">
                {artist_img_html}
                <div>
                    <div class="artist-name">{artist_name}</div>
                    <div class="artist-sub">Matched artist: {matched_artist}</div>
                </div>
            </div>

            <div class="section">
                <h2>Solo Songs</h2>
                <div class="songs-grid">
                    {build_song_cards(grouped_songs["solo"])}
                </div>
            </div>

            <div class="section">
                <h2>Main Artist Songs</h2>
                <div class="songs-grid">
                    {build_song_cards(grouped_songs["main"])}
                </div>
            </div>

            <div class="section">
                <h2>Featured Songs</h2>
                <div class="songs-grid">
                    {build_song_cards(grouped_songs["feature"])}
                </div>
            </div>

        </div>
    </body>
    </html>
    """

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)

    print("HTML file created:", os.path.abspath(output_file))

# ==============================
# 6. RUN
# ==============================
show_artist_songs_html("travis scott")

HTML file created: c:\HW3-Extra credit\artist_results.html


In [ ]:
show_artist_songs_html("sweater weather", output_file="song_results.html")

HTML file created: c:\HW3-Extra credit\song_results.html
